### Test TM-align
### Julian Moran, Vinicius Furlan
### 2026-02-25

In [9]:
import logging
import os

from Bio.PDB import PDBIO
from dataclasses import dataclass
from dotenv import load_dotenv
from tmtools import tm_align
from tmtools.io import get_residue_data, get_structure

import polars as pl

# Env
load_dotenv("../.env")
INSTALL_PATH = os.environ["INSTALL_PATH"]

# logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [5]:
# ────────────────────────────────────────────────────────────
#      Args
# ────────────────────────────────────────────────────────────

OUT_DIR = f"{INSTALL_PATH}/results/test_tm-align"
DATA_FILE_PROTEIN_PAIRS = f"{INSTALL_PATH}/results/explore_aJain_iei_pipeline/defense_proteins_filt_n=297.tsv"
COL_QUERY_PROTEIN_IDS = "defense_system_protein_id"
COL_TARGET_PROTEIN_IDS = "protein_id"

In [6]:
# ────────────────────────────────────────────────────────────
#      In
# ────────────────────────────────────────────────────────────

df_protein_pairs = pl.read_csv(
    DATA_FILE_PROTEIN_PAIRS,
    separator="\t",
    has_header=True
)

df_protein_pairs

accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio,GRIID_gene
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,f64,f64,str
"""WP_074585635.1""","""Eleos""","""Eleos""","""Bacillus mobilis,Bacillus cere…","""GCF_021655235_NZ_AP022971_Eleo…","""GCF_021655175.1_NZ_AP022953_00…","""GCF_021655235.1_NZ_AP022971_00…",14,189,"""A0A1Y6A5H7""","""A0A1I1GWW8""","""A0A1I1GWW8""","""A0A6Q8PGV8""",1.2200e-96,1,0.125,750,501,0.0,34,783,5,578,1.3890e-11,191,"""unreviewed""","""A0A6Q8PGV8_HUMAN""","""Mitofusin 2""","""MFN2""","""Homo sapiens (Human)""",null,null,"""MFN2""",null,"""UP000005640: Chromosome 1""","""mitochondrial fusion [GO:00080…","""mitochondrial outer membrane […","""mitochondrial outer membrane […","""GTP binding [GO:0005525]; GTPa…","""GO:0003924; GO:0005525; GO:000…","""cd09912; DLP_2; 1.;""","""1.20.5.110:FF:000012; Mitofusi…","""1.20.5.110; -; 1.;""3.40.50.300…","""IPR045063; Dynamin_N.;""IPR0068…","""PTHR10465:SF1; MITOFUSIN-2; 1.…","""PF00350; Dynamin_N; 1.;""PF0479…","""PS51718; G_DYNAMIN_2; 1.;""",null,null,"""34.0..783.0""",750,801,"""93..342""",100.0,33.33,93.63,"""['Dynamin-type G']""","""['ECO:0000259|PROSITE:PS51718'…","""5.0..578.0""",574,579,"""53..185""",100.0,23.17,99.14,"""['G']""","""['ECO:0000259|Pfam:PF01926']""",0.722846,574.0,0.991364,"""Yes"""
"""WP_074585635.1""","""Eleos""","""Eleos""","""Bacillus mobilis,Bacillus cere…","""GCF_021655235_NZ_AP022971_Eleo…","""GCF_021655175.1_NZ_AP022953_00…","""GCF_021655235.1_NZ_AP022971_00…",14,189,"""A0A556BFS8""","""A0A1I1GWW8""","""A0A1I1GWW8""","""A0A6Q8PFJ4""",1.2200e-96,1,0.128,696,494,0.0,40,735,7,574,2.5940e-12,198,"""unreviewed""","""A0A6Q8PFJ4_HUMAN""","""Mitofusin 2""","""MFN2""","""Homo sapiens (Human)""",null,null,"""MFN2""",null,"""UP000005640: Chromosome 1""","""mitochondrial fusion [GO:00080…","""mitochondrial outer membrane […","""mitochondrial outer membrane […","""GTP binding [GO:0005525]; GTPa…","""GO:0003924; GO:0005525; GO:000…","""cd09912; DLP_2; 1.;""","""3.40.50.300:FF:000214; Mitofus…","""1.20.5.110; -; 1.;""3.40.50.300…","""IPR045063; Dynamin_N.;""IPR0068…","""PTHR10465:SF1; MITOFUSIN-2; 1.…","""PF00350; Dynamin_N; 1.;""PF0479…","""PS51718; G_DYNAMIN_2; 1.;""",null,null,"""40.0..735.0""",696,808,"""93..342""",100.0,35.92,86.14,"""['Dynamin-type G']""","""['ECO:0000259|PROSITE:PS51718'…","""7.0..574.0""",568,579,"""352..545""",100.0,34.15,98.1,"""['Dynamin-like helical']""","""['ECO:0000259|Pfam:PF18709']""",0.716584,568.0,0.981002,"""Yes"""
"""WP_000434627.1""","""Eleos""","""Eleos""","""Escherichia coli""","""GCF_016776005_NZ_CP068823_Eleo…","""GCF_016775985.1_NZ_CP068827_02…","""GCF_020883255.1_NZ_CP086618_00…",27,176,"""Q9RFR9""","""A0A1I1GWW8""","""A0A1I1GWW8""","""O95140""",1.2200e-96,1,0.116,703,497,0.0,29,731,4,566,3.102

In [ ]:
# ────────────────────────────────────────────────────────────
#      TM-align
# ────────────────────────────────────────────────────────────

get_tm-alignment() -> 